# Linear static analysis

A thin isotropic cylinder, closed and supported at both ends, is loaded by
a radial point force at mid-length. The example builds the mesh, assembles
the linear stiffness matrix $\boldsymbol{K}_{C0}$ with `update_KC0`,
imposes the boundary conditions, solves for the displacements and plots
the radial displacement $w$. The same model is checked by the test
`tests/test_linear_point_load.py`.

**Theory.** For small displacements the internal forces are
$\boldsymbol{K}_{C0} \boldsymbol{u}$. With the constrained degrees of
freedom zero, the system of the unknown ones is, see
[Analyses](../analyses.rst),

$$
\boldsymbol{K}_{uu} \boldsymbol{u}_u = \boldsymbol{F}_u
$$

with $\boldsymbol{K}_{uu}$ symmetric and positive definite once the
rigid-body motions are constrained. The element is the rectangle of
Bogner, Fox and Schmit [Bogner1965] on the developed surface of the
cylinder, with Donnell-type kinematics [Donnell1933], see
[The BFSC element on the cylinder](../element.rst).

In [1]:
import numpy as np
from numpy import isclose, pi
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import spsolve
from composites import isotropic_plate

from bfsccylinder import (BFSCCylinder, update_KC0, DOF, DOUBLE, INT,
                          KC0_SPARSE_SIZE, KG_SPARSE_SIZE, M_SPARSE_SIZE)
from bfsccylinder.quadrature import get_points_weights
from bfsccylinder.utils import assign_constant_ABD

from IPython.display import Markdown


def table(rows, fmt='{:.5g}'):
    """Markdown table of a list of dicts with the same keys"""
    keys = list(rows[0])
    cell = lambda v: fmt.format(v) if isinstance(v, float) else str(v)
    lines = ['| ' + ' | '.join(keys) + ' |', '|' + '---|'*len(keys)]
    lines += ['| ' + ' | '.join(cell(r[k]) for k in keys) + ' |' for r in rows]
    return Markdown('\n'.join(lines))

## Geometry, material and mesh

$R = 0.4$ m, $L = 0.8$ m, $h = 1$ mm, aluminium, $E = 70$ GPa and
$\nu = 0.33$. `isotropic_plate` gives the laminate stiffnesses
$\boldsymbol{A}$ and $\boldsymbol{D}$ of the wall. The mesh has 11 nodes
along the axis, an odd number such that a circle of nodes lies at
$x = L/2$, and 30 around the circumference; the last column of the node
grid takes the numbers of the first, closing the cylinder, see
[Assembling a model](../assembling.rst).

In [2]:
L = 0.8  # m
R = 0.4  # m
b = 2*pi*R  # m, developed circumference
E = 70e9  # Pa
nu = 0.33
h = 0.001  # m
prop = isotropic_plate(thickness=h, E=E, nu=nu)

nx = 11
ny = 30

# node numbers on a grid of nx by ny + 1, the last column closing the cylinder
nids = 1 + np.arange(nx*(ny+1))
nids_mesh = nids.reshape(nx, ny+1)
nids_mesh[:, -1] = nids_mesh[:, 0]
nids = np.unique(nids_mesh)
nid_pos = dict(zip(nids, np.arange(len(nids))))

# coordinates of the nodes on the developed mid-surface
xlin = np.linspace(0, L, nx)
ylin = np.linspace(0, b, ny+1)[:-1]
xmesh, ymesh = np.meshgrid(xlin, ylin, indexing='ij')
x = xmesh.flatten()
y = ymesh.flatten()

# the four nodes of each element, counterclockwise from (x_i, y_j)
n1s = nids_mesh[:-1, :-1].flatten()
n2s = nids_mesh[1:, :-1].flatten()
n3s = nids_mesh[1:, 1:].flatten()
n4s = nids_mesh[:-1, 1:].flatten()
num_elements = len(n1s)
N = DOF*nx*ny
print('nodes:', nx*ny, ' elements:', num_elements, ' degrees of freedom:', N)

nodes: 330  elements: 300  degrees of freedom: 3300


## Elements and stiffness matrix

One `BFSCCylinder` per element, with the positions `c1` to `c4` of its
nodes in the global vectors, its sides `lex` and `ley`, the radius and the
laminate stiffnesses. `update_KC0` writes the entries of each element in
three sparse arrays, which `coo_matrix` sums into the global matrix.

In [3]:
nint = 4
points, weights = get_points_weights(nint=nint)

elements = []
for k, (n1, n2, n3, n4) in enumerate(zip(n1s, n2s, n3s, n4s)):
    shell = BFSCCylinder(nint)
    shell.n1, shell.n2, shell.n3, shell.n4 = n1, n2, n3, n4
    shell.c1 = DOF*nid_pos[n1]
    shell.c2 = DOF*nid_pos[n2]
    shell.c3 = DOF*nid_pos[n3]
    shell.c4 = DOF*nid_pos[n4]
    shell.R = R
    shell.lex = L/(nx - 1)
    shell.ley = b/ny
    assign_constant_ABD(shell, prop)
    shell.init_k_KC0 = k*KC0_SPARSE_SIZE
    shell.init_k_KG = k*KG_SPARSE_SIZE
    shell.init_k_M = k*M_SPARSE_SIZE
    elements.append(shell)

Kr = np.zeros(KC0_SPARSE_SIZE*num_elements, dtype=INT)
Kc = np.zeros(KC0_SPARSE_SIZE*num_elements, dtype=INT)
Kv = np.zeros(KC0_SPARSE_SIZE*num_elements, dtype=DOUBLE)
for shell in elements:
    update_KC0(shell, points, weights, Kr, Kc, Kv)
KC0 = coo_matrix((Kv, (Kr, Kc)), shape=(N, N)).tocsc()
print('KC0:', KC0.shape, KC0.nnz, 'nonzeros')

KC0: (3300, 3300) 279000 nonzeros


## Boundary conditions and load


Both ends are simply supported: $u$, $v$ and $w$ are constrained at the
nodes of $x = 0$ and $x = L$. Between two nodes of an end, $w$ is the cubic
of the nodal $w$ and $w_{,y}$, so with $w_{,y}$ free $w = 0$ holds exactly
only at the nodes; the deviation between them is small and vanishes with
mesh refinement, see [Assembling a model](../assembling.rst).


The radial force of $-100$ N, inwards since a positive $w$ points
outwards, is the entry of the global force vector at the position of $w$
of the node $x = L/2$, $y = \pi R$.

In [4]:
bk = np.zeros(N, dtype=bool)
ends = isclose(x, 0) | isclose(x, L)
for dof in (0, 3, 6):  # u, v, w
    bk[dof::DOF] = ends
bu = ~bk

f = np.zeros(N)
fmid = -100.  # N
loaded = isclose(x, L/2) & isclose(y, b/2)
f[6::DOF][loaded] = fmid
print('constrained:', bk.sum(), ' unknown:', bu.sum())

constrained: 180  unknown: 3120


## Solution

The reduced system is solved with a sparse direct solver and the solution
is expanded to the size of the model.

In [5]:
Kuu = KC0[bu][:, bu]
u = np.zeros(N)
u[bu] = spsolve(Kuu, f[bu])
w = u[6::DOF].reshape(nx, ny)
print(f'largest w = {w.max():.6e} m, smallest w = {w.min():.6e} m')

largest w = 2.350104e-04 m, smallest w = -7.055332e-04 m


In [6]:
fig, ax = plt.subplots(figsize=(7, 3.4))
c = ax.contourf(ymesh*1e3, xmesh*1e3, w*1e3, levels=50, cmap='coolwarm')
fig.colorbar(c, ax=ax, label='$w$ (mm)')
ax.plot(b/2*1e3, L/2*1e3, 'kx')
ax.set_xlabel('$y$ (mm), around the circumference')
ax.set_ylabel('$x$ (mm)')
ax.set_aspect('equal')
plt.show()

## Comparison

No published solution exists for this model; the references are the
regression values stored in `tests/test_linear_point_load.py`, results of
the same model, which guard the implementation against changes.

In [7]:
ref = {'largest w': 0.00023523450899127333, 'smallest w': -0.000705462681909122}
res = {'largest w': w.max(), 'smallest w': w.min()}
tab = table([{'result': k, 'value (m)': res[k], 'reference (m)': ref[k],
                       'source': 'regression value of the test',
                       'rel. diff. (%)': 100*(res[k] - ref[k])/ref[k]}
                      for k in ref])
tab

| result | value (m) | reference (m) | source | rel. diff. (%) |
|---|---|---|---|---|
| largest w | 0.00023501 | 0.00023523 | regression value of the test | -0.095268 |
| smallest w | -0.00070553 | -0.00070546 | regression value of the test | 0.010002 |

## References

- **[Bogner1965]** Bogner, F. K., Fox, R. L., and Schmit, L. A., The generation of
  inter-element-compatible stiffness and mass matrices by the use of
  interpolation formulas, Proceedings of the Conference on Matrix Methods in
  Structural Mechanics, AFFDL-TR-66-80, Wright-Patterson Air Force Base,
  Dayton, Ohio, 26-28 October 1965, pp. 395-444.
- **[Donnell1933]** Donnell, L. H., Stability of thin-walled tubes under torsion, NACA
  Report 479, 1933.